# Day 15 — Gradio demo app (Colab test, then Hugging Face Spaces)
Two tabs over the same retrieval pipeline:
* **Ask about one paper** — the frozen within-paper agent (Day 7 v2), with an optional **multi-hop** switch (Day 14).
* **Ask across papers** — the cross-paper agent (Day 13, v2 prompt).

Answers show clickable **[E#]** citations that open the cited passage, the papers found (with arXiv links) and a trace
of what the agent did (search queries, sufficiency check, tokens, time, which LLM answered).
**Quota protection:** precomputed examples (built from saved, judged results: no API calls), an answer cache, a
per-session and per-day cap on live questions, and **Groq → Gemini fallback** when Groq's daily quota is used up.

In [2]:
# WHAT: Mounts Google Drive and makes the research-engine folder the working directory.
# WHY:  All commands below run from the project root. Run this first in every new session.
from google.colab import drive
drive.mount('/content/drive')

import os, glob
PROJECT = glob.glob('/content/drive/MyDrive/research-engine')[0]
os.chdir(PROJECT)
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/research-engine


In [3]:
# WHAT: Installs the project's packages plus Gradio (the same version the Space uses) and the Hugging Face client.
# WHY:  A fresh Colab session doesn't have them.
!pip install -q -r requirements.txt pytest "gradio==6.30.0" huggingface_hub

In [4]:
# WHAT: Copies the Groq and Gemini keys from Colab Secrets into environment variables and checks for a GPU.
# WHY:  The app reads keys from the environment (like the Space will). Gemini is only the fallback, so a missing
#       GEMINI_API_KEY is fine; then the app uses Groq only.
from google.colab import userdata
import torch
for k in ("GROQ_API_KEY", "GEMINI_API_KEY"):
    try:
        os.environ[k] = userdata.get(k).strip()
        print(k, "loaded")
    except Exception:
        print(k, "not set (optional for Gemini)")
print("GPU:", torch.cuda.is_available())

GROQ_API_KEY loaded
GEMINI_API_KEY loaded
GPU: True


In [5]:
# WHAT: Runs the whole test suite (62 tests: everything before + the app's rendering, engine and LLM fallback).
# WHY:  Confirms the app logic works before starting it. Expect "62 passed".
!python -m pytest -q

..............................................................           [100%]
62 passed in 1.27s


## Step 1 — precomputed examples (no API calls)

In [6]:
# WHAT: Picks demo examples from results that are already saved and judged: within-paper answers rated correct AND
#       faithful, multi-hop answers that made a follow-up search, and cross-paper answers rated faithful.
#       Writes app/examples.json.
# WHY:  Visitors get instant answers that cost no quota. Re-run this after Day 14's run + judge are complete
#       to add the multi-hop examples.
!python -m app.build_examples

7 within-paper examples (2 multi-hop), 5 cross-paper examples -> /content/drive/MyDrive/research-engine/app/examples.json (223 KB)
  [paper] What measures are used for evaluation?  (Performance Comparison of Crowdworkers a)
  [paper] In what way is the offensive dataset not biased by topic, dialect or t  (Arabic Offensive Language on Twitter: An)
  [paper] What background knowledge do they leverage?  (Robustly Leveraging Prior Knowledge in T)
  [paper] Why are current ELS's not sufficiently effective?  (Entity Commonsense Representation for Ne)
  [paper] Which datasets do they use?  (Neural Collective Entity Linking)
  [paper +hop] In what way is the offensive dataset not biased by topic, dialect or t  (Arabic Offensive Language on Twitter: An)
  [paper +hop] What were their distribution results?  (Characterizing Political Fake News in Tw)
  [corpus] Which papers use the MultiWOZ dataset in their experiments, and for which dialogue task?
  [corpus] Which papers report experiments on th

## Step 2 — how fast is the reranker on a CPU (like the free Space)?

In [13]:
# WHAT: Measures dev evidence Recall@5 for three smaller rerankers on all 887 dev questions (no LLM calls).
#       For comparison: the frozen bge-reranker-v2-m3 gets 0.793 on dev, and hybrid search with NO reranker 0.747.
# WHY:  To choose the demo reranker by its measured quality. Results are saved under their own names; the frozen
#       result files are not touched.
!python -m eval.run_retrieval_eval --split dev --reranker cross-encoder/ms-marco-MiniLM-L-6-v2  | grep -E "^\| final|reranker="
!python -m eval.run_retrieval_eval --split dev --reranker cross-encoder/ms-marco-MiniLM-L-12-v2 | grep -E "^\| final|reranker="
!python -m eval.run_retrieval_eval --split dev --reranker BAAI/bge-reranker-base --max-length 256 | grep -E "^\| final|reranker="

Loading weights: 100% 199/199 [00:00<00:00, 5025.63it/s]
config.json: 100% 794/794 [00:00<00:00, 3.48MB/s]

model.safetensors: downloading bytes:  86% 78.1M/90.9M [00:00<00:00, 115MB/s, 5.82MB/s  ] 
model.safetensors: downloading bytes: 100% 86.0M/86.0M [00:01<00:00, 83.6MB/s, 8.26MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 88.3MB/s, 8.87MB/s  ]
Loading weights: 100% 105/105 [00:00<00:00, 4499.22it/s]
tokenizer_config.json: 100% 1.33k/1.33k [00:00<00:00, 4.38MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 25.5MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 90.9MB/s]
special_tokens_map.json: 100% 132/132 [00:00<00:00, 497kB/s]
Batches: 100% 237/237 [00:19<00:00, 12.17it/s]
scope=paper  split=dev  n=887  reranker=cross-encoder/ms-marco-MiniLM-L-6-v2  excluded=['abstract']
| final | 0.428 | 0.855 | 0.770 | 0.939 | 0.605 |
Loading weights: 100% 199/199 [00:00<00:00, 4695.50it/s]
config.json: 100% 791/791 [00:00<00:00, 4.17MB/s]

model.safetensors: downlo

In [14]:
# WHAT: Times each small reranker on CPU with all cores (30 passages = one-paper mode, 100 = cross-paper mode).
# WHY:  The demo needs a reranker that answers in a few seconds on the free 2-core Space.
!CUDA_VISIBLE_DEVICES="" python -m app.bench_cpu --model cross-encoder/ms-marco-MiniLM-L-6-v2 --sizes 30,100
!CUDA_VISIBLE_DEVICES="" python -m app.bench_cpu --model cross-encoder/ms-marco-MiniLM-L-12-v2 --sizes 30,100
!CUDA_VISIBLE_DEVICES="" python -m app.bench_cpu --model BAAI/bge-reranker-base --max-length 256 --sizes 30,100

cross-encoder/ms-marco-MiniLM-L-6-v2 | max_length 512 | device: CPU | threads: 2 | cpus: 2
Loading weights: 100% 105/105 [00:00<00:00, 5111.15it/s]
rerank  30 pairs:    3.9 s
rerank 100 pairs:   13.2 s
cross-encoder/ms-marco-MiniLM-L-12-v2 | max_length 512 | device: CPU | threads: 2 | cpus: 2
Loading weights: 100% 201/201 [00:00<00:00, 4646.01it/s]
rerank  30 pairs:    7.7 s
rerank 100 pairs:   25.7 s
BAAI/bge-reranker-base | max_length 256 | device: CPU | threads: 2 | cpus: 2
Loading weights: 100% 201/201 [00:00<00:00, 4590.45it/s]
rerank  30 pairs:   13.8 s
rerank 100 pairs:   43.0 s


In [7]:
# WHAT: Times the cross-encoder (bge-reranker-v2-m3) on 30, 50 and 100 passages with the GPU hidden. No LLM calls.
# WHY:  The free Hugging Face Space has no GPU. Within-paper mode reranks 30 passages, cross-paper mode 100; if 100
#       is too slow on CPU, the Space uses a lower APP_CROSS_RERANK_TOP for the demo only (the evaluation keeps 100).
!CUDA_VISIBLE_DEVICES="" python -m app.bench_cpu

device: CPU | threads: 1 | cpus: 2
Loading weights: 100% 393/393 [00:00<00:00, 4693.50it/s]
rerank  30 pairs: 105.3 s
rerank  50 pairs: 177.4 s
rerank 100 pairs: 323.5 s


## Step 3 — run the app here (public test link)
Click the **gradio.live** link that appears. Try an example in each tab, then a live question. When done, stop the
cell (■). If today's Groq quota is used up, live questions fall back to Gemini (or show a message); the examples
always work.

In [11]:
# WHAT: Starts the app on Colab's GPU with a temporary public link (valid ~72 h, only while this cell runs).
# WHY:  To check the whole app end to end before deploying it.
!python -m app.app --share

* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://b680b7576fe892ddc5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Loading weights: 100% 199/199 [00:00<00:00, 4455.62it/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.npy
Loading weights: 100% 393/393 [00:00<00:00, 5517.71it/s]
FAISS IndexFlatIP built in 0.03 s (28541 vectors, dim 384)
engine ready in 22 s | live LLM: True
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 3957, in block_thread
    time.sleep(0.1)
    ~~~~~~~~~~^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback

## Step 4 — deploy to Hugging Face Spaces (Wednesday)
Needs a Hugging Face account and a token with **write** access stored as the Colab secret **HF_TOKEN**
(huggingface.co → Settings → Access Tokens → New token → type "Write").

In [1]:
# WHAT: Creates the Space <your HF user>/research-evidence-engine and uploads code, examples, chunks and embeddings in
#       one commit; stores the Groq/Gemini keys as Space SECRETS (never in files) and sets the demo limits.
# WHY:  The public demo link for the README and CV. The Space then builds for ~5-10 minutes.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()
!python -m app.deploy_space --cross-rerank-top 100

NameError: name 'userdata' is not defined

## Results
* CPU reranker timing: …
* Space URL: …